# Burgers: error versus output dimension

We learn the solution operator $K: u_0 \mapsto u(\cdot, T)$ of the viscous Burgers' equation
$u_t + u u_x = \nu u_{xx}$ on $(0,1)$, $u(0,t) = u(1,t) = 0$, with $T = 0.2$.

* Inputs and outputs are sine coefficients in the basis $\sqrt 2 \sin(n\pi x)$; data come from the spectral Galerkin solver in `burgers_solver.py`.
* Input measure: $d_{\mathrm{in}} = 20$ independent coefficients, the $n$-th distributed as $\mathrm{Jacobi}(\alpha_n, \alpha_n)$. **As run:** $\alpha_n = (n-1)^2$, $n = 1, \dots, 20$ (so the first mode is uniform on $[-1,1]$).
* Approximation space: polynomial operators $\psi_m\, p_\lambda(u_0)$ with $\lambda$ in an anisotropic index set with weights $w_j$ = `linspace(1, 0.01, 20)` (the weights enter as an exponent $1/w_j$, see `owls.index_set`) and univariate degree at most 10.

For each $\nu \in \{1, 0.1, 0.01, 0.001\}$ we fit one operator with all 200 output modes (hyperbolic cross $k=60$, $M = \lfloor N\log N\rfloor$ optimal samples), then keep only its first $d_{\mathrm{out}}$ output coefficients. Because the least-squares problem decouples over output modes, this equals fitting with $d_{\mathrm{out}}$ outputs. We report the relative test error (1000 test inputs) against the full 200-mode solution. The black crosses mark the first $d_{\mathrm{out}}$ whose error is within 1% of the error at $d_{\mathrm{out}} = 192$.

**Status:** the data in `data/` are from the arXiv v1 run; this experiment was not rerun after the cleanup.

In [ ]:
RECOMPUTE = False  # True: rerun the experiment and overwrite data/ (see ../README.md for run times)
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
from burgers_solver import burgers_map, sine_basis_vandermonde

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters (as run)
NUS = [1, 0.1, 0.01, 0.001]
D_SOLVE = 200                     # solver modes; all are kept as outputs
HC_K = 60
MAX_SIZE = 10000
T = 0.2
M_TEST = 1000
DOUT_GRID = np.arange(2, 200, 10)
SEED = 0
dt_for = lambda nu: 1e-3 if nu >= 0.01 else 1e-4   # as run
if QUICK:
    NUS, HC_K, M_TEST, T, D_SOLVE, DOUT_GRID = [0.1], 5, 50, 0.01, 40, np.arange(2, 40, 10)

D_IN = 20                                             # input sine modes
MAX_DEGREE = 10                                       # max univariate polynomial degree
measures = [owls.Jacobi(a**2) for a in range(D_IN)]  # as run: alpha_n = (n - 1)^2 for n = 1..20
weights = np.linspace(1, 0.01, D_IN)                 # anisotropy weights

In [ ]:
if RECOMPUTE or QUICK:
    rel_errs = np.zeros((len(NUS), len(DOUT_GRID)))
    L = owls.index_set(D_IN, HC_K, "hc", weights, MAX_DEGREE, MAX_SIZE)
    N = len(L)
    for i, nu in enumerate(NUS):
        rng = np.random.default_rng([SEED, i])
        solver = lambda u0: burgers_map(u0, nu, T, dt_for(nu), D_SOLVE)
        model = owls.learn_operator(solver, measures, L, M=int(N * np.log(N)), optimal=True, rng=rng)
        x_test = owls.sample_rho(M_TEST, measures, rng)
        y_test = solver(x_test)
        V = model.vandermonde(x_test)
        for j, dd in enumerate(DOUT_GRID):
            y_pred = np.zeros_like(y_test)
            y_pred[:, :dd] = V @ model.C[:, :dd]     # keep the first dd output modes
            rel_errs[i, j] = owls.relative_bochner_error(y_pred, y_test)
        print(f"nu = {nu} done")
    np.save(os.path.join(DATA, "out_dim_dat.npy"), rel_errs)

In [ ]:
rel_errs = np.load(os.path.join(DATA, "out_dim_dat.npy"))

fig_width, fig_height = paper_style(width=0.5, aspect=0.6, fontsize=7, legend_fontsize=5)
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout="constrained")
colors, markers = ["darkred", "firebrick", "indianred", "lightcoral"], ["D", "^", "o", "*"]
for i, nu in enumerate(NUS):
    ax.plot(DOUT_GRID, rel_errs[i], marker=markers[i], linestyle="--", color=colors[i], label=rf"$\nu$ = {nu}", linewidth=1, markersize=3)
    # first d_out whose error is within 1% of the error at the largest d_out
    idx = np.argmin(np.abs(1 - (rel_errs[i] - rel_errs[i][-1]) / rel_errs[i][-1] * 100))
    ax.plot(DOUT_GRID[idx], rel_errs[i][idx], "kX", markersize=4.5, label=r"$1\%$ Rel. Error $d_{\mathrm{out}}$" if i == len(NUS) - 1 else None)
ax.set(yscale="log", xscale="log", xlabel=r"$d_{\mathrm{out}}$", ylabel=r"Relative $L^2_{\rho}$ Error")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], loc="best")
savefig(os.path.join(FIGS, "vb_truncation_error_with_viscosity.pdf"))
plt.show()